In [8]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

In [9]:
# Function to extract Product Title
def get_title(soup):

    try:
        # Outer Tag Object
        title = soup.find("span", attrs={"id":'productTitle'})

        # Inner NavigatableString Object
        title_value = title.text

        # Title as a string value
        title_string = title_value.strip()

    except AttributeError:
        title_string = ""

    return title_string

# Function to extract Product Price
def get_price(soup):

    try:
        price = soup.find("span", attrs={'id':'priceblock_ourprice'}).string.strip()

    except AttributeError:

        try:
            # If there is some deal price
            price = soup.find("span", attrs={'id':'priceblock_dealprice'}).string.strip()

        except:
            price = ""

    return price

# Function to extract Product Rating
def get_rating(soup):

    try:
        rating = soup.find("i", attrs={'class':'a-icon a-icon-star a-star-4-5'}).string.strip()

    except AttributeError:
        try:
            rating = soup.find("span", attrs={'class':'a-icon-alt'}).string.strip()
        except:
            rating = ""

    return rating

# Function to extract Number of User Reviews
def get_review_count(soup):
    try:
        review_count = soup.find("span", attrs={'id':'acrCustomerReviewText'}).string.strip()

    except AttributeError:
        review_count = ""

    return review_count

# Function to extract Availability Status
def get_availability(soup):
    try:
        available = soup.find("div", attrs={'id':'availability'})
        available = available.find("span").string.strip()

    except AttributeError:
        available = "Not Available"

    return available



In [16]:
if __name__ == '__main__':

    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US, en;q=0.5'
    }

    # The webpage URL
    URL = "https://www.amazon.com/s?k=playstation+4&ref=nb_sb_noss_2"

    # HTTP Request
    webpage = requests.get(URL, headers=HEADERS)

    # Soup Object containing all data
    soup = BeautifulSoup(webpage.content, "html.parser")

    # Fetch links as List of Tag Objects
    links = soup.find_all("a", attrs={'class': 'a-link-normal s-no-outline'})

    # Store the links
    links_list = []

    # Loop for extracting links from Tag Objects
    for link in links:
        links_list.append(link.get('href'))

    # Dictionary to store product details
    d = {"title": [], "price": [], "rating": [], "reviews": [], "availability": []}

    # Loop for extracting product details from each link
    for link in links_list:
        new_webpage = requests.get("https://www.amazon.com" + link, headers=HEADERS)
        new_soup = BeautifulSoup(new_webpage.content, "html.parser")

        # Extracting and appending product information
        d['title'].append(get_title(new_soup))
        d['price'].append(get_price(new_soup))
        d['rating'].append(get_rating(new_soup))
        d['reviews'].append(get_review_count(new_soup))
        d['availability'].append(get_availability(new_soup))

    # Create a DataFrame from the dictionary
    amazon_df = pd.DataFrame.from_dict(d)

    # Clean DataFrame
    amazon_df['title'].replace('', np.nan, inplace=True)
    amazon_df = amazon_df.dropna(subset=['title'])

    # Save to CSV
    amazon_df.to_csv("amazon_data.csv", header=True, index=False)

<ipython-input-16-655b498d8e6c>:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  amazon_df['title'].replace('', np.nan, inplace=True)


In [17]:
amazon_df

,title,price,rating,reviews,availability
0,Sony PlayStation 4 500GB Console (Renewed),,4.0 out of 5 stars,"2,912 ratings",Only 1 left in stock - order soon.
1,Sony PlayStation 4 Slim Limited Edition 1TB Ga...,,4.2 out of 5 stars,"1,528 ratings",In Stock
2,"Playstation SONY 4, 500GB Slim System [CUH-221...",,4.2 out of 5 stars,502 ratings,
3,PlayStation 4 Slim 1TB Console,,4.7 out of 5 stars,"15,779 ratings",Only 10 left in stock - order soon.
4,Sony Playstation PS4 1TB Black Console,,4.3 out of 5 stars,"1,590 ratings",
5,Zeust PlayStation 4 Slim 1TB Console Bundle - ...,,2.6 out of 5 stars,3 ratings,In Stock
6,Sony PlayStation 4 PRO 1TB Gaming Console - Bl...,,4.7 out of 5 stars,206 ratings,Only 16 left in stock - order soon.
7,Sony PlayStation 4 500GB Premium Bundle (Renewed),,3.7 out of 5 stars,37 ratings,In Stock
8,PlayStation®5 Digital Edition (slim),,4.7 out of 5 stars,"9,862 ratings",In Stock
9,PlayStation 4 Slim 1TB Limited Edition Console...,,4.7 out of 5 stars,2 ratings,Only 1 left in stock - order soon.
